# 06 — Predict FIFA World Cup 2026

Uses trained models + final Elo + the official 2026 groups to predict every group match, then runs a **10,000-rollout Monte Carlo** simulation of the group stage to estimate qualification probabilities for each team.

In [ ]:
import sys, pathlib, joblib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from itertools import combinations
from collections import defaultdict
from src.wc2026_config import GROUPS, HOST_COUNTRIES, normalize, all_teams
from src.elo import k_for, HOME_ADV

clf      = joblib.load(ROOT / 'models' / 'xgb_result.joblib')
pr_home  = joblib.load(ROOT / 'models' / 'poisson_home.joblib')
pr_away  = joblib.load(ROOT / 'models' / 'poisson_away.joblib')
FEATURES = joblib.load(ROOT / 'models' / 'feature_list.joblib')
elo      = pd.read_csv(ROOT / 'data' / 'processed' / 'final_elo.csv').set_index('team').elo.to_dict()
matches  = pd.read_parquet(ROOT / 'data' / 'processed' / 'matches_features.parquet')
matches['home_team'] = matches.home_team.map(normalize)
matches['away_team'] = matches.away_team.map(normalize)
print('models + elo loaded. elo entries:', len(elo))


In [ ]:
# Most recent form snapshot per team (from feature table).
def latest_form(team):
    h = matches[matches.home_team == team].tail(1)
    a = matches[matches.away_team == team].tail(1)
    cand = pd.concat([h.assign(side='h'), a.assign(side='a')])
    if cand.empty:
        return None
    row = cand.sort_values('date').tail(1).iloc[0]
    if row.side == 'h':
        return dict(form5_pts=row.home_form5_pts, form5_gf=row.home_form5_gf, form5_ga=row.home_form5_ga,
                    form10_pts=row.home_form10_pts, form10_gf=row.home_form10_gf, form10_ga=row.home_form10_ga)
    return dict(form5_pts=row.away_form5_pts, form5_gf=row.away_form5_gf, form5_ga=row.away_form5_ga,
                form10_pts=row.away_form10_pts, form10_gf=row.away_form10_gf, form10_ga=row.away_form10_ga)

form_cache = {t: latest_form(t) for t in all_teams()}
missing = [t for t,v in form_cache.items() if v is None]
print('teams with no form data:', missing)


In [ ]:
def build_row(home, away, tournament='FIFA World Cup', neutral=True):
    fh = form_cache.get(home) or dict(form5_pts=1, form5_gf=1, form5_ga=1, form10_pts=1, form10_gf=1, form10_ga=1)
    fa = form_cache.get(away) or dict(form5_pts=1, form5_gf=1, form5_ga=1, form10_pts=1, form10_gf=1, form10_ga=1)
    h_elo = elo.get(home, 1500.0)
    a_elo = elo.get(away, 1500.0)
    # crude H2H proxy = recent meetings in matches table
    h2h = matches[((matches.home_team==home)&(matches.away_team==away)) |
                  ((matches.home_team==away)&(matches.away_team==home))].tail(5)
    hw=dr=aw=0; gf=ga=0.0
    for _, r in h2h.iterrows():
        if r.home_team == home:
            hg, ag = r.home_score, r.away_score
        else:
            hg, ag = r.away_score, r.home_score
        if hg>ag: hw+=1
        elif hg<ag: aw+=1
        else: dr+=1
        gf += hg; ga += ag
    n = max(len(h2h), 1)
    return {
        'home_elo': h_elo, 'away_elo': a_elo, 'elo_diff': h_elo-a_elo,
        'home_form5_pts': fh['form5_pts'], 'home_form5_gf': fh['form5_gf'], 'home_form5_ga': fh['form5_ga'],
        'away_form5_pts': fa['form5_pts'], 'away_form5_gf': fa['form5_gf'], 'away_form5_ga': fa['form5_ga'],
        'home_form10_pts': fh['form10_pts'], 'home_form10_gf': fh['form10_gf'], 'home_form10_ga': fh['form10_ga'],
        'away_form10_pts': fa['form10_pts'], 'away_form10_gf': fa['form10_gf'], 'away_form10_ga': fa['form10_ga'],
        'h2h_home_wins': hw, 'h2h_draws': dr, 'h2h_away_wins': aw,
        'h2h_home_gf': gf/n, 'h2h_home_ga': ga/n,
        'tournament_k': k_for(tournament),
        'neutral_int': int(neutral), 'home_rest': 4, 'away_rest': 4,
    }

def predict(home, away, neutral=True):
    row = build_row(home, away, neutral=neutral)
    X = pd.DataFrame([row])[FEATURES]
    p = clf.predict_proba(X)[0]
    eh = float(pr_home.predict(X)[0])
    ea = float(pr_away.predict(X)[0])
    return {'home_win': float(p[0]), 'draw': float(p[1]), 'away_win': float(p[2]),
            'expected_home_goals': eh, 'expected_away_goals': ea}

predict('Brazil', 'Morocco')


In [ ]:
# Predict every group match
rows = []
for g, teams in GROUPS.items():
    for a, b in combinations(teams, 2):
        # Treat host countries as home when they play in their own group; else neutral.
        host_a = a in HOST_COUNTRIES
        host_b = b in HOST_COUNTRIES
        neutral = not (host_a or host_b)
        home, away = (a, b) if host_a or not host_b else (b, a)
        p = predict(home, away, neutral=neutral)
        rows.append(dict(group=g, home=home, away=away, neutral=neutral, **p))
preds = pd.DataFrame(rows)
(ROOT / 'outputs').mkdir(exist_ok=True)
preds.to_csv(ROOT / 'outputs' / 'group_stage_predictions.csv', index=False)
preds.head(12)


In [ ]:
# Monte Carlo group-stage simulator using Poisson expected goals.
rng = np.random.default_rng(42)
N_SIM = 10_000

# Precompute expected goals per (home, away) ordered pair in each group.
lookup = {(r.home, r.away): (r.expected_home_goals, r.expected_away_goals) for r in preds.itertuples()}

advance_count = defaultdict(int)   # advance as top-2
winner_count  = defaultdict(int)
third_count   = defaultdict(int)   # finished 3rd (best-third candidates)

for _ in range(N_SIM):
    for g, teams in GROUPS.items():
        pts = {t: 0 for t in teams}
        gf  = {t: 0 for t in teams}
        ga  = {t: 0 for t in teams}
        for a, b in combinations(teams, 2):
            if (a, b) in lookup:
                eh, ea = lookup[(a, b)]; home, away = a, b
            else:
                eh, ea = lookup[(b, a)]; home, away = b, a
            hg = rng.poisson(max(eh, 0.05))
            ag = rng.poisson(max(ea, 0.05))
            gf[home] += hg; ga[home] += ag; gf[away] += ag; ga[away] += hg
            if hg > ag: pts[home] += 3
            elif hg < ag: pts[away] += 3
            else: pts[home] += 1; pts[away] += 1
        ranking = sorted(teams, key=lambda t: (pts[t], gf[t]-ga[t], gf[t]), reverse=True)
        winner_count[ranking[0]] += 1
        advance_count[ranking[0]] += 1
        advance_count[ranking[1]] += 1
        third_count[ranking[2]]   += 1

summary = []
for g, teams in GROUPS.items():
    for t in teams:
        summary.append({
            'group': g, 'team': t,
            'win_group_%':   round(100 * winner_count[t]  / N_SIM, 1),
            'top2_%':        round(100 * advance_count[t] / N_SIM, 1),
            'third_place_%': round(100 * third_count[t]   / N_SIM, 1),
        })
summary_df = pd.DataFrame(summary).sort_values(['group','top2_%'], ascending=[True, False])
summary_df.to_csv(ROOT / 'outputs' / 'group_stage_simulation.csv', index=False)
summary_df


### Next steps

- Add **player-level features** (FBref club stats, Transfermarkt market value) per docs1.md §C–E.
- Build the **bracket generator** for the new Round of 32 (docs3.md) and extend the simulator end-to-end to champion probabilities.
- Ensemble: blend XGBoost with CatBoost + a neural net (docs1.md Layer 6).
- Daily refresh: re-run notebook 03 every day until kickoff to keep Elo + form current.